# LeetCode #1391: Check if There is a Valid Path in a Grid

https://leetcode.com/problems/check-if-there-is-a-valid-path-in-a-grid/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS from every cell)** | $O((nm)^2)$ | $O(nm)$ |
| **Optimal: BFS with Street Connectivity ★** | $O(nm)$ | $O(nm)$ |

---

## Understanding the Methods

### Brute Force (DFS from every cell)
Try DFS from (0,0), check neighbour connections at each step. Without careful direction tables, this repeats work.

### Optimal: BFS with Street Connectivity ★
Encode each of the 6 street types as a pair of (direction, incoming direction). BFS from (0,0): a move from cell A to neighbour B is valid only if A's street connects toward B **and** B's street connects back toward A. Reach (m-1,n-1) → true.

**Constraints:**
* $1 \le m, n \le 300$
* `grid[i][j]` is one of $\{1,2,3,4,5,6\}$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public bool HasValidPath(int[][] grid) {
        int rows = grid.Length, cols = grid[0].Length;
        // For each street type, which directions does it connect to?
        // Directions: 0=left, 1=right, 2=up, 3=down
        var connects = new int[][] {
            new[]{1,0},   // 1: left-right (connects left and right)
            new[]{2,3},   // 2: up-down
            new[]{0,3},   // 3: left-down
            new[]{1,3},   // 4: right-down
            new[]{0,2},   // 5: left-up
            new[]{1,2},   // 6: right-up
        };
        // Direction deltas and opposite directions
        int[] dr = {0,0,-1,1};
        int[] dc = {-1,1,0,0};
        int[] opp = {1,0,3,2}; // opposite of left=right, right=left, up=down, down=up

        var visited = new bool[rows, cols];
        visited[0,0] = true;
        var q = new Queue<(int r, int c)>();
        q.Enqueue((0,0));

        while (q.Count > 0) {
            var (r, c) = q.Dequeue();
            if (r == rows-1 && c == cols-1) return true;
            int type = grid[r][c] - 1;
            // Try each direction this street type connects to
            foreach (int dir in connects[type]) {
                int nr = r + dr[dir], nc = c + dc[dir];
                if (nr < 0 || nr >= rows || nc < 0 || nc >= cols) continue;
                if (visited[nr, nc]) continue;
                // Neighbour must connect back toward us
                int ntype = grid[nr][nc] - 1;
                bool connects_back = Array.IndexOf(connects[ntype], opp[dir]) >= 0;
                if (connects_back) { visited[nr,nc] = true; q.Enqueue((nr,nc)); }
            }
        }
        return visited[rows-1, cols-1];
    }
}

### Python

In [ ]:
from collections import deque

class Solution:
    def hasValidPath(self, grid: list[list[int]]) -> bool:
        rows, cols = len(grid), len(grid[0])
        # Each street type connects two specific directions
        # Directions: 0=left, 1=right, 2=up, 3=down
        connects = {
            1: (0, 1),   # horizontal
            2: (2, 3),   # vertical
            3: (0, 3),   # left-down corner
            4: (1, 3),   # right-down corner
            5: (0, 2),   # left-up corner
            6: (1, 2),   # right-up corner
        }
        dr = [0, 0, -1, 1]
        dc = [-1, 1, 0, 0]
        opposite = [1, 0, 3, 2]  # opposite direction index

        visited = [[False]*cols for _ in range(rows)]
        visited[0][0] = True
        q = deque([(0, 0)])

        while q:
            r, c = q.popleft()
            if r == rows-1 and c == cols-1:
                return True
            for d in connects[grid[r][c]]:
                nr, nc = r + dr[d], c + dc[d]
                if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc]:
                    # Neighbour must connect back toward the current cell
                    if opposite[d] in connects[grid[nr][nc]]:
                        visited[nr][nc] = True
                        q.append((nr, nc))

        return visited[rows-1][cols-1]

### Go

In [ ]:
func hasValidPath(grid [][]int) bool {
	rows, cols := len(grid), len(grid[0])
	// Street connections: each type connects exactly two directions
	// 0=left,1=right,2=up,3=down
	connects := [7][2]int{
		{},       // placeholder (types are 1-indexed)
		{0, 1},   // 1: left-right
		{2, 3},   // 2: up-down
		{0, 3},   // 3: left-down
		{1, 3},   // 4: right-down
		{0, 2},   // 5: left-up
		{1, 2},   // 6: right-up
	}
	dr := [4]int{0, 0, -1, 1}
	dc := [4]int{-1, 1, 0, 0}
	opp := [4]int{1, 0, 3, 2}

	visited := make([][]bool, rows)
	for i := range visited { visited[i] = make([]bool, cols) }
	visited[0][0] = true
	q := [][2]int{{0,0}}

	for len(q) > 0 {
		cur := q[0]; q = q[1:]
		r, c := cur[0], cur[1]
		if r == rows-1 && c == cols-1 { return true }
		for _, d := range connects[grid[r][c]] {
			nr, nc := r+dr[d], c+dc[d]
			if nr < 0 || nr >= rows || nc < 0 || nc >= cols || visited[nr][nc] { continue }
			// Check that neighbour's street connects back to us
			nb := connects[grid[nr][nc]]
			if nb[0] == opp[d] || nb[1] == opp[d] {
				visited[nr][nc] = true
				q = append(q, [2]int{nr, nc})
			}
		}
	}
	return visited[rows-1][cols-1]
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn has_valid_path(grid: Vec<Vec<i32>>) -> bool {
        let (rows, cols) = (grid.len(), grid[0].len());
        // directions: 0=left,1=right,2=up,3=down
        let connects: [[usize;2];7] = [
            [0,0],   // unused
            [0,1],   // 1: left-right
            [2,3],   // 2: up-down
            [0,3],   // 3: left-down
            [1,3],   // 4: right-down
            [0,2],   // 5: left-up
            [1,2],   // 6: right-up
        ];
        let dr: [i32;4] = [0,0,-1,1];
        let dc: [i32;4] = [-1,1,0,0];
        let opp: [usize;4] = [1,0,3,2];

        let mut visited = vec![vec![false; cols]; rows];
        visited[0][0] = true;
        let mut q = VecDeque::from([(0usize, 0usize)]);

        while let Some((r, c)) = q.pop_front() {
            if r == rows-1 && c == cols-1 { return true; }
            let t = grid[r][c] as usize;
            for &d in &connects[t] {
                let nr = r as i32 + dr[d];
                let nc = c as i32 + dc[d];
                if nr < 0 || nr >= rows as i32 || nc < 0 || nc >= cols as i32 { continue; }
                let (nr, nc) = (nr as usize, nc as usize);
                if visited[nr][nc] { continue; }
                let nt = grid[nr][nc] as usize;
                // Neighbour connects back toward us
                if connects[nt][0] == opp[d] || connects[nt][1] == opp[d] {
                    visited[nr][nc] = true;
                    q.push_back((nr, nc));
                }
            }
        }
        visited[rows-1][cols-1]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid=[[2,4,3],[6,5,2]]`
A valid path exists: (0,0)→(1,0)→(1,1)→(1,2)→(0,2). BFS finds this path. Answer: **true**.

### 2. Slightly Complex
**Input:** `grid=[[1,2,1],[1,2,1]]`
Streets 2 (vertical) run top-to-bottom but type 1 (horizontal) can't connect downward. No valid path. Answer: **false**.

### 3. Edge Case: Time Factor
**Input:** $300 \times 300$ grid where every cell is type 1 (horizontal).
BFS visits each row's cells left-to-right; vertical connections are blocked. Only $O(nm) = O(90000)$ operations total.

### 4. Edge Case: Space Factor
**Input:** $300 \times 300$ fully connected grid.
The visited array and BFS queue together use $O(nm)$ memory — at most 90 000 entries.

### 5. Almost-Impossible but Plausible
**Input:** $1 \times 1$ grid, `grid=[[1]]`.
Start equals destination. BFS immediately checks termination and returns **true** without expanding any neighbours.